# 01 — Signal Processing

This notebook processes binary `.Q00` conductivity fluctuation measurements
and creates spectral features for thin-film Bi, Cr, and Fe samples.

It uses the same processing pipeline as `src/signal_processing.py`.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = BASE_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from signal_processing import (
    DATA_DIR,
    PROCESSED_DIR,
    DATASET_PATH,
    FILE_METADATA,
    MATERIAL_NAMES,
    MATERIAL_COLORS,
    read_q00_file,
    split_into_samples,
    compute_spectrum_features,
    process_q00_file,
    process_file,
    build_combined_dataset,
)

print("Data directory:", DATA_DIR)
print("Processed directory:", PROCESSED_DIR)
print("Combined dataset:", DATASET_PATH)

## Input data

In [ ]:
q00_files = sorted(DATA_DIR.glob("*.Q00"))

print(f"Found {len(q00_files)} .Q00 files")

for file_path in q00_files:
    print(file_path.name)

## File metadata

Each file has known material, measurement year, irradiation status,
measurement period, and sampling rate.

In [ ]:
metadata_df = pd.DataFrame(FILE_METADATA).T

metadata_df

## Read one binary file

The reader extracts calibration coefficients, record metadata, and raw ADC codes.

In [ ]:
example_file_stem = "BI202C01"
example_file = DATA_DIR / f"{example_file_stem}.Q00"

records = list(read_q00_file(example_file))

print(f"File: {example_file.name}")
print(f"Number of records: {len(records)}")
print(f"Points in first record: {len(records[0]['codes'])}")

records[0]

## Split signal into overlapping samples

The default configuration uses 128-point samples with 50% overlap.

In [ ]:
sample_length = 128
overlap_mode = 2

samples = split_into_samples(
    codes=records[0]["codes"],
    sample_length=sample_length,
    overlap_mode=overlap_mode,
)

print(f"Number of samples: {len(samples)}")
print(f"Sample shape: {samples[0].shape}")

## Process one file

For every sample, the pipeline removes a linear trend, applies a Blackman
window, computes the spectrum, and extracts `gamma` and `energy`.

In [ ]:
df_example = process_file(
    input_dir=DATA_DIR,
    file_stem=example_file_stem,
    output_dir=PROCESSED_DIR,
    sampling_rate=FILE_METADATA[example_file_stem]["sampling_rate_hz"],
)

df_example.head()

In [ ]:
df_example[["gamma", "energy", "log_energy"]].describe()

## Build combined dataset

This processes all available `.Q00` files and saves both individual feature
files and the combined dataset.

In [ ]:
df = build_combined_dataset()

df.head()

## Dataset overview

In [ ]:
df.info()

In [ ]:
df["material"].value_counts()

In [ ]:
df.groupby(["material", "irradiated"])[["gamma", "log_energy"]].agg(
    ["count", "mean", "std"]
)

## Spectral feature distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    data=df[df["material"] != "XXX"],
    x="gamma",
    hue="material",
    kde=True,
    ax=axes[0],
)
axes[0].set_title("Distribution of spectral exponent γ")
axes[0].set_xlabel("Spectral exponent γ")
axes[0].set_ylabel("Number of samples")
axes[0].grid(True, alpha=0.3)

sns.histplot(
    data=df[df["material"] != "XXX"],
    x="log_energy",
    hue="material",
    kde=True,
    ax=axes[1],
)
axes[1].set_title("Distribution of log10 spectral energy")
axes[1].set_xlabel("Log10 spectral energy")
axes[1].set_ylabel("Number of samples")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Stable vs irradiated films

Irradiated films are not used for training. They are compared with stable
films to identify possible radiation-induced changes in noise features.

In [ ]:
plt.figure(figsize=(11, 7))

sns.scatterplot(
    data=df[df["material"] != "XXX"],
    x="gamma",
    y="log_energy",
    hue="material",
    style="irradiated",
    s=45,
    alpha=0.75,
)

plt.xlabel("Spectral exponent γ")
plt.ylabel("Log10 spectral energy")
plt.title(
    "Conductivity Fluctuation Features:\n"
    "Stable vs Irradiated Thin-Film Materials"
)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Check saved dataset

In [ ]:
df_loaded = pd.read_csv(DATASET_PATH)

print(f"Saved dataset: {DATASET_PATH}")
print(f"Rows: {len(df_loaded)}")
print(f"Files: {df_loaded['file'].nunique()}")

df_loaded.head()

## Output files

In [ ]:
processed_files = sorted(PROCESSED_DIR.glob("*_features.csv"))

print(f"Individual feature files: {len(processed_files)}")

for file_path in processed_files:
    print(file_path.name)

print("\nCombined dataset:")
print(DATASET_PATH)